# 38. Prefill/Decode Scheduling | Prefill/Decode 调度
**难度：** Medium | **环境：** CPU-first | **标签：** `推理优化`, `Serving`, `Chunked Prefill`, `PD Disaggregation` | **目标人群：** 推理优化学习者

---

## 本节导读

长 prompt 到达时，完整 Prefill 可能占住一整轮批次，使正在生成的请求等待更久。Chunked Prefill 先把未处理输入切成可让出的执行块；当 Prefill 与 Decode 的资源压力长期不同，再考虑把它们组织成独立服务池。

本节建立“请求压力 → 分块推进 → 分池与交接 → 服务决策”的链路。它先解释调度如何减少请求间的相互阻塞，再判断 PD 分离的吞吐和延迟收益能否覆盖交接成本。

**关键词：** `chunked prefill`, `continuous batching`, `PD disaggregation`, `handoff`

## 前置阅读

**导语：** 先能区分 Prefill 的突发输入处理与 Decode 的持续 KV 读取，再观察未命中 suffix 为什么要按可让出的粒度进入批次。

- [24. Prefix Cache Matching and Reuse | Prefix Cache 匹配与复用](./24_Prefix_Cache_Matching_and_Reuse.ipynb)
- [INF-ITERATION-BUDGET. Serving 迭代成本与批次容量上界](./INF-ITERATION-BUDGET_Serving_Iteration_Cost_and_Batch_Bounds.ipynb)
- [36. Decode Scheduling | Decode 调度](./36_Decode_Scheduling.ipynb)
- [37. KV Cache Lifecycle and Capacity Control | KV Cache 生命周期与容量控制](./37_KV_Cache_Lifecycle_and_Capacity_Control.ipynb)
- [INF-CONTINUOUS-BATCHING. Continuous Batching | 连续批处理](./INF-CONTINUOUS-BATCHING_Continuous_Batching.ipynb)

### Step 1: 请求为什么需要同时考虑 Prefill 与 Decode

一次请求先处理完整输入，再逐 token 生成输出。长 prompt 往往带来集中的 Prefill 计算和访存；长生成则长期占用 KV Cache，并对 Decode 等待敏感。Continuous Batching 已负责在每轮重组请求成员，本节进一步决定 ready 请求中的 Prefill suffix 和 Decode 工作如何共享 `max_num_batched_tokens`：本轮 Prefill chunk 的 token 数与 Decode 请求数之和不能超过这份预算。

| 请求形态 | 更突出的压力 | 首先观察什么 |
|---|---|---|
| 长输入、短生成 | Prefill 计算、输入访存与批次占用 | TTFT、Prefill 队列、Decode 抖动 |
| 短输入、长生成 | KV Cache 驻留、逐步 Decode 与排队 | TPOT、KV 使用量、Decode 队列 |
| 输入和生成都长 | 两类压力叠加 | E2E、P95/P99、池利用率 |

![Continuous Batching：Chunked Prefill 与 Decode 的调度时间线](../docs/public/02_PyTorch_Algorithms/38_prefill_decode_timeline.svg)

### Step 2: Chunked Prefill 如何保留 Decode 的准入机会

Prefix Cache 未命中的 suffix 仍要 Prefill。Chunked Prefill 将它切成能进入本轮 token 预算的块：每完成一个 chunk，调度器便回到 ready 集合，决定继续当前请求还是先服务等待中的 Decode。分块既限制单轮工作量，也增加重新选择请求的机会；prefix 是否命中仍由缓存机制决定。

| 调度对象 | 含义 | 影响 |
|---|---|---|
| `suffix` | 本次仍需处理的未命中输入 | 决定新增 Prefill 工作量 |
| `chunk_size` | 单次进入 Prefill 的最大 token 数 | 必须服从剩余 token budget，控制单轮占用和让出频率 |
| `chunk_plan` | suffix 的有序分块计划 | 让调度器在块边界重新选请求 |
| `iteration_plan` | 本轮 Decode token 与 Prefill chunk 的组合 | 保证总 scheduled tokens 不越过预算 |
| `yield_point` | 一个 chunk 完成后的可重排位置 | 缓解长 Prefill 对 Decode 的阻塞 |

![Chunked Prefill：后缀分块与可让出执行计划](../docs/public/02_PyTorch_Algorithms/38_chunked_prefill_schedule.svg)

### Step 3: 何时需要进入 Prefill/Decode 分离

若分块后长输入仍持续抬高 Decode 等待时间，单池调度已经不能消除资源争用。此时本节只输出进入 Prefill/Decode 分离评估的条件；资源池能力、状态交接与失败回退由 39 节负责。

| 观察结果 | 本节动作 | 后续入口 |
|---|---|---|
| 分块后 Decode 等待恢复 | 保留单池分块调度 | 继续观察 TTFT、TPOT 与吞吐 |
| 分块后 Decode 仍被长 Prefill 阻塞 | 记录单池基线并进入分离评估 | 39 节资源池与状态交接 |
| 分离后的交接代价抵消收益 | 回退单池或调整资源配置 | 39 节失败回退与服务分层 |

![Prefill 与 Decode 的拆分流程：分类、交接与指标比较](../docs/public/02_PyTorch_Algorithms/38_pd_disaggregation.svg)

### Step 4：实现 CPU 调度账本与决策

题目区只实现本节拥有的两项机制：将未命中输入切成有序块，并在单轮 token 预算内组合 Prefill 与 Decode 工作。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
|---|---|---|---|
| TODO 1 / `build_chunked_prefill_plan` | 保持顺序地切分未命中输入 | token 不丢失、不重排；尾块可以更短 | 顺序、尾块与非法块大小 |
| TODO 2 / `plan_iteration_budget` | 在单轮预算内组合 Decode token 与 Prefill chunk | 三种策略均满足 token 守恒和预算上限 | 策略差异、预算守恒与非法输入 |


In [ ]:
from typing import Dict, List


In [ ]:
def build_chunked_prefill_plan(suffix_tokens: List[int], chunk_size: int) -> List[List[int]]:
    """把未命中 suffix 切成有序 chunk；尾块可以小于 chunk_size。"""
    # ==========================================
    # TODO 1（分块计划）：保持 token 顺序，并允许尾块短于 chunk_size。
    # chunks = ???
    # ==========================================
    raise NotImplementedError


def plan_iteration_budget(prefill_suffixes: Dict[str, int], decode_request_ids: List[str], max_num_batched_tokens: int, max_prefill_chunk_tokens: int, policy: str = 'balanced', decode_reserve_tokens: int = 1) -> Dict[str, object]:
    """在单轮 token 预算内分配 Decode token 与 Prefill chunk。"""
    # ==========================================
    # TODO 2（阶段预算）：实现三种策略，并保持 used_tokens 不超过单轮预算。
    # plan = ???
    # ==========================================
    raise NotImplementedError

In [ ]:
# 机制测试：分别检查输入分块与单轮 token 预算。
def test_chunked_prefill_plan():
    """验证 suffix 顺序、尾块和非法 chunk_size。"""
    assert build_chunked_prefill_plan([9, 10, 11, 12, 13], chunk_size=2) == [[9, 10], [11, 12], [13]]
    try:
        build_chunked_prefill_plan([1], chunk_size=0)
    except ValueError:
        pass
    else:
        raise AssertionError('非正 chunk_size 必须拒绝')


def test_iteration_budget_policies():
    """验证三种阶段策略的成员差异与 token 预算守恒。"""
    suffixes = {'p1': 5, 'p2': 2}
    decode_ids = ['d1', 'd2']

    balanced = plan_iteration_budget(suffixes, decode_ids, 4, 2, policy='balanced', decode_reserve_tokens=1)
    assert balanced['decode_request_ids'] == ['d1']
    assert balanced['prefill_chunks'] == [{'request_id': 'p1', 'tokens': 2}, {'request_id': 'p2', 'tokens': 1}]

    decode_first = plan_iteration_budget(suffixes, decode_ids, 3, 2, policy='decode_first')
    assert decode_first['decode_request_ids'] == ['d1', 'd2']
    assert decode_first['prefill_chunks'] == [{'request_id': 'p1', 'tokens': 1}]

    prefill_first = plan_iteration_budget(suffixes, decode_ids, 3, 2, policy='prefill_first')
    assert prefill_first['decode_request_ids'] == []
    assert prefill_first['prefill_chunks'] == [{'request_id': 'p1', 'tokens': 2}, {'request_id': 'p2', 'tokens': 1}]

    for plan in (balanced, decode_first, prefill_first):
        assert plan['used_tokens'] <= plan['max_num_batched_tokens']
        assert plan['used_tokens'] + plan['remaining_tokens'] == plan['max_num_batched_tokens']


def test_iteration_budget_contract():
    """验证非法策略、非正预算与负 suffix 会被拒绝。"""
    invalid_cases = [
        ({'p': 1}, [], 0, 1, 'balanced'),
        ({'p': 1}, [], 2, 1, 'unknown'),
        ({'p': -1}, [], 2, 1, 'balanced'),
    ]
    for suffixes, decode_ids, budget, chunk, policy in invalid_cases:
        try:
            plan_iteration_budget(suffixes, decode_ids, budget, chunk, policy=policy)
        except ValueError:
            pass
        else:
            raise AssertionError('非法阶段预算输入必须拒绝')
def test_prefill_decode_scheduling():
    """汇总分块与单轮预算机制测试。"""
    try:
        test_chunked_prefill_plan()
        test_iteration_budget_policies()
        test_iteration_budget_contract()
        print('✅ Prefill/Decode 调度机制测试通过：分块顺序、策略差异和预算守恒均通过。')
    except NotImplementedError:
        raise
    except (NameError, AttributeError, TypeError, ValueError, AssertionError) as error:
        raise NotImplementedError('请先完成 TODO 代码或检查字段名！') from error


test_prefill_decode_scheduling()


## 参考代码与解析

### 代码


In [ ]:
def build_chunked_prefill_plan(suffix_tokens: List[int], chunk_size: int) -> List[List[int]]:
    """把未命中 suffix 切成有序 chunk；尾块可以小于 chunk_size。"""
    # TODO 1：生成分块预填充计划
    if chunk_size <= 0:
        raise ValueError('chunk_size 必须为正数')
    suffix = list(suffix_tokens)
    return [suffix[start:start + chunk_size] for start in range(0, len(suffix), chunk_size)]


def plan_iteration_budget(prefill_suffixes: Dict[str, int], decode_request_ids: List[str], max_num_batched_tokens: int, max_prefill_chunk_tokens: int, policy: str = 'balanced', decode_reserve_tokens: int = 1) -> Dict[str, object]:
    """在单轮 token 预算内分配 Decode token 与 Prefill chunk。"""
    # TODO 2：用阶段优先级分配预算，并保持 used_tokens 不超过单轮上限
    if max_num_batched_tokens <= 0 or max_prefill_chunk_tokens <= 0:
        raise ValueError('token 预算与 Prefill chunk 上限必须为正数')
    if decode_reserve_tokens < 0:
        raise ValueError('decode_reserve_tokens 不能为负数')
    if policy not in {'decode_first', 'prefill_first', 'balanced'}:
        raise ValueError(f'不支持的阶段策略: {policy}')
    if any(tokens < 0 for tokens in prefill_suffixes.values()):
        raise ValueError('Prefill suffix 长度不能为负数')
    if len(set(decode_request_ids)) != len(decode_request_ids):
        raise ValueError('Decode 请求 ID 不能重复')

    remaining = max_num_batched_tokens
    scheduled_decode: List[str] = []
    prefill_chunks: List[Dict[str, object]] = []

    def schedule_decode(limit: int | None = None) -> None:
        nonlocal remaining
        pending = [request_id for request_id in decode_request_ids if request_id not in scheduled_decode]
        take = min(len(pending), remaining) if limit is None else min(len(pending), remaining, limit)
        scheduled_decode.extend(pending[:take])
        remaining -= take

    def schedule_prefill() -> None:
        nonlocal remaining
        for request_id, suffix_tokens in prefill_suffixes.items():
            if remaining == 0:
                break
            chunk_tokens = min(suffix_tokens, max_prefill_chunk_tokens, remaining)
            if chunk_tokens > 0:
                prefill_chunks.append({'request_id': request_id, 'tokens': chunk_tokens})
                remaining -= chunk_tokens

    if policy == 'decode_first':
        schedule_decode()
        schedule_prefill()
    elif policy == 'prefill_first':
        schedule_prefill()
        schedule_decode()
    else:
        schedule_decode(limit=decode_reserve_tokens)
        schedule_prefill()
        schedule_decode()

    used_tokens = max_num_batched_tokens - remaining
    return {
        'policy': policy,
        'max_num_batched_tokens': max_num_batched_tokens,
        'decode_request_ids': scheduled_decode,
        'prefill_chunks': prefill_chunks,
        'used_tokens': used_tokens,
        'remaining_tokens': remaining,
    }

### 解析

#### TODO 1：生成分块预填充计划

- **机制**：只切分前缀缓存未命中的输入，并在每个块结束后形成新的调度边界。
- **不变量**：token 顺序和总数不变，块大小为正，只有尾块可以短于上限。
- **测试证据**：测试同时覆盖顺序、尾块和非法块大小。

#### TODO 2：分配单轮阶段预算

- **机制**：解码优先、预填充优先和均衡策略改变同一轮中两类工作的先后顺序。
- **不变量**：每个解码请求消耗一个 token 预算，预填充块和解码 token 的总量不超过 `max_num_batched_tokens`。
- **测试证据**：三种策略产生不同成员组合，但都满足预算守恒；非法策略、非正预算和负输入长度会被拒绝。

### Step 5：真实后端实验——整段预填充与分块预填充

CPU 题目确认输入分块与单轮 token 预算；真实后端先建立活跃解码流，再注入长预填充与短探针请求，比较整段预填充和分块预填充对解码停顿与探针首 token 延迟的影响。PD 分池不在单卡实验中模拟。

| 学习问题 | CPU 已确认 | 真实后端要确认 |
|---|---|---|
| 输入如何进入单轮预算 | 顺序分块与 token 守恒 | 后端是否按固定 token 上限切分预填充 |
| 分块保护了什么 | 块边界提供重新调度机会 | 活跃解码流的最大输出间隔是否缩短 |
| 新请求能否及时进入 | 阶段预算决定本轮成员 | 长预填充之后到达的短探针 TTFT 是否下降 |
| 分块是否有代价 | 更频繁的调度会增加开销 | 请求吞吐是否明显下降 |

#### 5.1 环境与配对运行方式

本实验使用 `INF-VLLM-011-CU128`。首次运行先按[推理服务环境 Quickstart](../topic_discussion/inference_optimization/quickstart.md#inf-vllm-quickstart)安装环境，并依次启动整段预填充和分块预填充服务。两次运行保持相同 `PAIR_LABEL`，只修改 `SERVICE_VARIANT`；Notebook 不同时启动两个服务。

| 配置项 | 整段预填充基线 | 分块预填充候选 |
|---|---|---|
| 模型与最大长度 | `Qwen/Qwen3-0.6B`，4096 | 相同 |
| 分块开关 | 关闭 | 开启 |
| 单轮 token 上限 | 后端随整段输入调度 | 512 |
| 到达序列 | 4 条活跃解码请求 → 1 条 2048-token 长输入 → 1 条短探针 | 相同到达时间与输出上限 |


In [5]:
# 5.1 每次只运行一个服务变体；两次运行共享 PAIR_LABEL。
RUN_MODE = "dry_run"  # 默认值 dry_run：只登记计划；服务就绪后改为 real_backend。
PAIR_LABEL = "default"  # 默认值 default：正式配对实验改为唯一标签。
SERVICE_VARIANT = "whole_prefill"  # 默认值 whole_prefill：完成后改为 chunked_prefill。

ENVIRONMENT_PROFILE = "INF-VLLM-011-CU128"  # Quickstart 中登记的服务环境。
BACKEND_BASE_URL = "http://127.0.0.1:8000"  # 当前已启动服务地址。
MODEL_ID = "Qwen/Qwen3-0.6B"  # 两个服务变体使用同一模型。
MODEL_REVISION = "main"  # 5.2 从本地缓存解析实际 commit。
ACTIVE_DECODE_REQUESTS = 4  # 默认值 4：先建立四条持续解码流。
ACTIVE_DECODE_PROMPT_TOKENS = 128  # 默认值 128：活跃解码请求的输入长度。
ACTIVE_DECODE_MAX_NEW_TOKENS = 256  # 默认值 256：强制保持足够长的解码窗口。
LONG_PREFILL_PROMPT_TOKENS = 2048  # 默认值 2048：制造预填充干扰。
LONG_PREFILL_MAX_NEW_TOKENS = 16  # 默认值 16：避免长请求的解码阶段主导结果。
SHORT_PROBE_PROMPT_TOKENS = 128  # 默认值 128：探针请求的输入长度。
SHORT_PROBE_MAX_NEW_TOKENS = 32  # 默认值 32：获得稳定的首 token 与完成时间。
LONG_PREFILL_ARRIVAL_DELAY_S = 0.20  # 默认值 0.20：等待活跃请求进入解码后再注入长输入。
SHORT_PROBE_ARRIVAL_DELAY_S = 0.22  # 默认值 0.22：让探针紧随长输入进入。
REQUEST_COUNT = ACTIVE_DECODE_REQUESTS + 2  # 四条解码流、一个长输入和一个短探针。
CONCURRENCY = REQUEST_COUNT  # 所有角色均可按计划到达后端。
WARMUP_STEPS = 1  # 每个服务变体开始前的预热次数。
CHUNK_TOKEN_BUDGET = 512  # 候选服务的 --max-num-batched-tokens。
RESULT_DIR_RELATIVE = "02_PyTorch_Algorithms/benchmarks/results/inf_chunked_prefill"


#### 5.2 执行当前服务变体并保存阶段结果

当前运行只测一个服务变体。完成后停止服务，按 Quickstart 切换后端配置，再修改 `SERVICE_VARIANT` 重跑本节；模型、请求顺序和 `PAIR_LABEL` 不变。


In [6]:
# 5.2 构造固定 token 长度请求，执行当前服务变体并保存阶段 JSON。
import json

import torch
from transformers import AutoTokenizer
from tools.inference_project_runtime import locate_repo_root
from tools.notebook_runtime import package_version, resolve_model_commit
from tools.serving_mechanism_runtime import percentile, run_arrival_schedule

if RUN_MODE not in {"dry_run", "real_backend"}:
    raise ValueError("RUN_MODE 必须是 dry_run 或 real_backend；服务实验不能使用 real_gpu")
if SERVICE_VARIANT not in {"whole_prefill", "chunked_prefill"}:
    raise ValueError("SERVICE_VARIANT 必须是 whole_prefill 或 chunked_prefill")
root = locate_repo_root()
result_dir = root / RESULT_DIR_RELATIVE
stage_path = result_dir / f"chunked_prefill_{PAIR_LABEL}_{SERVICE_VARIANT}.json"
pair_path = result_dir / f"chunked_prefill_{PAIR_LABEL}_pair.json"
declared_backend = {
    "whole_prefill": {"enable_chunked_prefill": False, "max_num_batched_tokens": None},
    "chunked_prefill": {"enable_chunked_prefill": True, "max_num_batched_tokens": CHUNK_TOKEN_BUDGET},
}[SERVICE_VARIANT]
stage = {
    "schema_version": "inference-chunked-prefill-stage/v2",
    "semantic_id": "INF-CHUNKED-PREFILL",
    "environment_profile": ENVIRONMENT_PROFILE,
    "run_mode": RUN_MODE, "pair_label": PAIR_LABEL, "service_variant": SERVICE_VARIANT,
    "framework": {"torch": torch.__version__, "vllm": package_version("vllm")},
    "backend_config": declared_backend,
    "workload": {
        "id": "inf-chunked-prefill-interference-v2", "model_id": MODEL_ID,
        "requested_revision": MODEL_REVISION, "resolved_commit": None,
        "active_decode_requests": ACTIVE_DECODE_REQUESTS,
        "active_decode_prompt_tokens": ACTIVE_DECODE_PROMPT_TOKENS,
        "active_decode_max_new_tokens": ACTIVE_DECODE_MAX_NEW_TOKENS,
        "long_prefill_prompt_tokens": LONG_PREFILL_PROMPT_TOKENS,
        "long_prefill_max_new_tokens": LONG_PREFILL_MAX_NEW_TOKENS,
        "short_probe_prompt_tokens": SHORT_PROBE_PROMPT_TOKENS,
        "short_probe_max_new_tokens": SHORT_PROBE_MAX_NEW_TOKENS,
        "long_prefill_arrival_delay_s": LONG_PREFILL_ARRIVAL_DELAY_S,
        "short_probe_arrival_delay_s": SHORT_PROBE_ARRIVAL_DELAY_S,
        "request_count": REQUEST_COUNT, "concurrency": CONCURRENCY,
    },
    "report": None, "mechanism_metrics": None,
    "failure": None, "evidence_level": "dry_run",
}

if RUN_MODE == "real_backend":
    try:
        commit, source = resolve_model_commit(MODEL_ID, MODEL_REVISION, local_files_only=True)
        stage["workload"]["resolved_commit"] = commit
        stage["workload"]["revision_source"] = source
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=commit, local_files_only=True)
        seed_text = "分块预填充通过限制单轮输入工作量，为等待中的解码请求保留调度机会。"
        def fixed_token_prompt(target_tokens, instruction):
            suffix_ids = tokenizer("\n" + instruction, add_special_tokens=False)["input_ids"]
            if len(suffix_ids) >= target_tokens:
                raise ValueError("目标 token 数必须大于指令后缀长度")
            body_budget = target_tokens - len(suffix_ids)
            body_ids = tokenizer(seed_text * (target_tokens // 8 + 16), add_special_tokens=False)["input_ids"][:body_budget]
            return tokenizer.decode(body_ids + suffix_ids, skip_special_tokens=True)
        prompts = [
            {
                "prompt": fixed_token_prompt(ACTIVE_DECODE_PROMPT_TOKENS, f"活跃解码 {index}：持续列出机制要点。"),
                "max_tokens": ACTIVE_DECODE_MAX_NEW_TOKENS, "arrival_delay_s": 0.0,
                "request_role": "active_decode", "ignore_eos": True,
            }
            for index in range(ACTIVE_DECODE_REQUESTS)
        ]
        prompts.extend([
            {
                "prompt": fixed_token_prompt(LONG_PREFILL_PROMPT_TOKENS, "请总结长输入中的核心机制。"),
                "max_tokens": LONG_PREFILL_MAX_NEW_TOKENS,
                "arrival_delay_s": LONG_PREFILL_ARRIVAL_DELAY_S, "request_role": "long_prefill",
            },
            {
                "prompt": fixed_token_prompt(SHORT_PROBE_PROMPT_TOKENS, "短探针：给出一句结论。"),
                "max_tokens": SHORT_PROBE_MAX_NEW_TOKENS,
                "arrival_delay_s": SHORT_PROBE_ARRIVAL_DELAY_S, "request_role": "short_probe",
            },
        ])
        stage["workload"]["client_content_token_counts"] = [
            len(tokenizer(item["prompt"], add_special_tokens=False)["input_ids"]) for item in prompts
        ]
        report = run_arrival_schedule(
            base_url=BACKEND_BASE_URL, model=MODEL_ID, prompts=prompts,
            schedule="rolling_admission", concurrency=CONCURRENCY, max_tokens=ACTIVE_DECODE_MAX_NEW_TOKENS,
            warmup=WARMUP_STEPS, timeout_s=300.0,
        )
        successful = [item for item in report["requests"] if item.get("ok")]
        active_rows = [item for item in successful if item.get("request_role") == "active_decode"]
        long_rows = [item for item in successful if item.get("request_role") == "long_prefill"]
        probe_rows = [item for item in successful if item.get("request_role") == "short_probe"]
        if len(active_rows) != ACTIVE_DECODE_REQUESTS or len(long_rows) != 1 or len(probe_rows) != 1:
            raise RuntimeError("请求角色未全部成功，不能形成分块预填充机制证据")
        active_gap_p95 = [item["stream_gap_ms"]["p95"] for item in active_rows if item["stream_gap_ms"]["p95"] is not None]
        active_gap_max = [item["stream_gap_ms"]["max"] for item in active_rows if item["stream_gap_ms"]["max"] is not None]
        if len(active_gap_p95) != ACTIVE_DECODE_REQUESTS or len(active_gap_max) != ACTIVE_DECODE_REQUESTS:
            raise RuntimeError("活跃解码流缺少连续 token 间隔，不能判断预填充阻塞")
        stage["report"] = report
        stage["mechanism_metrics"] = {
            "short_probe_ttft_ms": probe_rows[0]["ttft_ms"],
            "long_prefill_ttft_ms": long_rows[0]["ttft_ms"],
            "active_decode_stream_gap_p95_ms": percentile(active_gap_p95, 95),
            "active_decode_stream_gap_max_ms": max(active_gap_max),
        }
        stage["evidence_level"] = "real_backend_mechanism_stage"
    except Exception as exc:
        stage.update({
            "failure": {"type": type(exc).__name__, "message": str(exc)},
            "evidence_level": "failed_real_backend_attempt",
        })

result_dir.mkdir(parents=True, exist_ok=True)
stage_path.write_text(json.dumps(stage, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(stage, ensure_ascii=False, indent=2))


{
  "schema_version": "inference-chunked-prefill-stage/v2",
  "semantic_id": "INF-CHUNKED-PREFILL",
  "environment_profile": "INF-VLLM-011-CU128",
  "run_mode": "real_backend",
  "pair_label": "rtx5070ti_chunked_interference_v3_reverse",
  "service_variant": "chunked_prefill",
  "framework": {
    "torch": "2.8.0+cu128",
    "vllm": "0.11.0"
  },
  "backend_config": {
    "enable_chunked_prefill": true,
    "max_num_batched_tokens": 512
  },
  "workload": {
    "id": "inf-chunked-prefill-interference-v2",
    "model_id": "Qwen/Qwen3-0.6B",
    "requested_revision": "main",
    "resolved_commit": "c1899de289a04d12100db370d81485cdf75e47ca",
    "active_decode_requests": 4,
    "active_decode_prompt_tokens": 128,
    "active_decode_max_new_tokens": 256,
    "long_prefill_prompt_tokens": 2048,
    "long_prefill_max_new_tokens": 16,
    "short_probe_prompt_tokens": 128,
    "short_probe_max_new_tokens": 32,
    "long_prefill_arrival_delay_s": 0.2,
    "short_probe_arrival_delay_s": 0.22,
  

#### 5.3 汇总两个服务变体

只有同一 `PAIR_LABEL` 下的两个阶段结果都成功，才比较短探针 TTFT、活跃解码流输出间隔、吞吐和失败数。


In [4]:
# 5.3 读取两个阶段 JSON 并形成配对结论；重启 Kernel 后不依赖 5.2 的内存状态。
import json
from tools.inference_project_runtime import locate_repo_root

result_dir = locate_repo_root() / RESULT_DIR_RELATIVE
pair_path = result_dir / f"chunked_prefill_{PAIR_LABEL}_pair.json"
stage_paths = {name: result_dir / f"chunked_prefill_{PAIR_LABEL}_{name}.json" for name in ("whole_prefill", "chunked_prefill")}
missing = [name for name, path in stage_paths.items() if not path.exists()]
pair = {
    "schema_version": "inference-chunked-prefill-pair/v3",
    "semantic_id": "INF-CHUNKED-PREFILL", "pair_label": PAIR_LABEL,
    "baseline": None, "candidate": None, "comparison": None, "decision_checks": None,
    "failure": None, "evidence_level": "incomplete_pair", "decision": "run_missing_variant",
}
if missing:
    pair["failure"] = {"type": "MissingVariant", "message": f"仍需运行：{missing}"}
else:
    pair["baseline"] = json.loads(stage_paths["whole_prefill"].read_text(encoding="utf-8"))
    pair["candidate"] = json.loads(stage_paths["chunked_prefill"].read_text(encoding="utf-8"))
    failures = [item["failure"] for item in (pair["baseline"], pair["candidate"]) if item["failure"]]
    identity_fields = (
        "id", "model_id", "resolved_commit", "active_decode_requests",
        "active_decode_prompt_tokens", "active_decode_max_new_tokens",
        "long_prefill_prompt_tokens", "long_prefill_max_new_tokens",
        "short_probe_prompt_tokens", "short_probe_max_new_tokens",
        "long_prefill_arrival_delay_s", "short_probe_arrival_delay_s",
        "request_count", "concurrency", "client_content_token_counts",
    )
    mismatches = [field for field in identity_fields if pair["baseline"]["workload"].get(field) != pair["candidate"]["workload"].get(field)]
    variants_valid = (
        pair["baseline"].get("service_variant") == "whole_prefill"
        and pair["candidate"].get("service_variant") == "chunked_prefill"
        and pair["baseline"]["backend_config"].get("enable_chunked_prefill") is False
        and pair["candidate"]["backend_config"].get("enable_chunked_prefill") is True
        and pair["candidate"]["backend_config"].get("max_num_batched_tokens") == CHUNK_TOKEN_BUDGET
    )
    if mismatches or not variants_valid:
        pair.update({"failure": {"type": "PairMismatch", "fields": mismatches, "variants_valid": variants_valid}, "decision": "reject_until_fixed"})
    elif failures:
        pair.update({"failure": {"type": "StageFailure", "details": failures}, "decision": "reject_until_fixed"})
    else:
        base_metrics = pair["baseline"]["report"]["metrics"]
        cand_metrics = pair["candidate"]["report"]["metrics"]
        base_mechanism = pair["baseline"]["mechanism_metrics"]
        cand_mechanism = pair["candidate"]["mechanism_metrics"]
        short_probe_ttft_ratio = cand_mechanism["short_probe_ttft_ms"] / max(base_mechanism["short_probe_ttft_ms"], 1e-12)
        active_gap_p95_ratio = cand_mechanism["active_decode_stream_gap_p95_ms"] / max(base_mechanism["active_decode_stream_gap_p95_ms"], 1e-12)
        active_gap_max_ratio = cand_mechanism["active_decode_stream_gap_max_ms"] / max(base_mechanism["active_decode_stream_gap_max_ms"], 1e-12)
        throughput_ratio = cand_metrics["request_throughput_per_s"] / max(base_metrics["request_throughput_per_s"], 1e-12)
        tpot_p95_ratio = cand_metrics["tpot_ms"]["p95"] / max(base_metrics["tpot_ms"]["p95"], 1e-12)
        all_succeeded = base_metrics["failed_requests"] == cand_metrics["failed_requests"] == 0
        pair["comparison"] = {
            "short_probe_ttft_ratio": short_probe_ttft_ratio,
            "active_decode_stream_gap_p95_ratio": active_gap_p95_ratio,
            "active_decode_stream_gap_max_ratio": active_gap_max_ratio,
            "request_throughput_ratio": throughput_ratio,
            "tpot_p95_ratio": tpot_p95_ratio,
            "all_requests_succeeded": all_succeeded,
        }
        pair["evidence_level"] = "real_backend_mechanism_benchmark"
        protects_decode_tail = active_gap_max_ratio <= 0.75 and active_gap_p95_ratio <= 1.10
        stable_throughput = throughput_ratio >= 0.90
        protects_probe = short_probe_ttft_ratio < 0.95
        pair["decision_checks"] = {
            "active_stream_gap_max_ratio_lte_0_75": active_gap_max_ratio <= 0.75,
            "active_stream_gap_p95_ratio_lte_1_10": active_gap_p95_ratio <= 1.10,
            "request_throughput_ratio_gte_0_90": stable_throughput,
            "short_probe_ttft_ratio_lt_0_95": protects_probe,
            "all_requests_succeeded": all_succeeded,
        }
        if all_succeeded and protects_decode_tail and stable_throughput:
            pair["decision"] = "accept_chunked_prefill" if protects_probe else "accept_decode_protection_with_probe_tradeoff"
        else:
            pair["decision"] = "tune"
pair_path.write_text(json.dumps(pair, ensure_ascii=False, indent=2), encoding="utf-8")
pair


{'schema_version': 'inference-chunked-prefill-pair/v3',
 'semantic_id': 'INF-CHUNKED-PREFILL',
 'pair_label': 'rtx5070ti_chunked_interference_v2',
 'baseline': {'schema_version': 'inference-chunked-prefill-stage/v2',
  'semantic_id': 'INF-CHUNKED-PREFILL',
  'environment_profile': 'INF-VLLM-011-CU128',
  'run_mode': 'real_backend',
  'pair_label': 'rtx5070ti_chunked_interference_v2',
  'service_variant': 'whole_prefill',
  'framework': {'torch': '2.8.0+cu128', 'vllm': '0.11.0'},
  'backend_config': {'enable_chunked_prefill': False,
   'max_num_batched_tokens': None},
  'workload': {'id': 'inf-chunked-prefill-interference-v2',
   'model_id': 'Qwen/Qwen3-0.6B',
   'requested_revision': 'main',
   'resolved_commit': 'c1899de289a04d12100db370d81485cdf75e47ca',
   'active_decode_requests': 4,
   'active_decode_prompt_tokens': 128,
   'active_decode_max_new_tokens': 256,
   'long_prefill_prompt_tokens': 2048,
   'long_prefill_max_new_tokens': 16,
   'short_probe_prompt_tokens': 128,
   'shor

#### 5.4 解释短请求保护与系统代价

纯预填充负载曾得到候选无收益的负例；加入活跃解码、长预填充和短探针的到达顺序后，正向与反向两轮配对均显示客户端观测的流式输出间隔和短探针首 Token 延迟下降。这里的流式间隔包含服务调度、流式传输与客户端读取时间，不等同于 GPU Kernel 的单 Token 执行时间。吞吐和每输出 Token 时间的改善幅度随运行状态变化，因此只把稳定方向归因于分块预填充，不把单轮加速比外推为服务容量结论。

| 配对与运行顺序 | 流式间隔 P95 / 最大值：基线 → 候选 | 短探针首 Token 延迟：基线 → 候选 | 吞吐比 | 每输出 Token 时间 P95 比 | 决策 |
|---|---:|---:|---:|---:|---|
| v2：整段 → 分块 | 14.47 / 89.93 → 8.46 / 22.51 ms | 125.40 → 98.21 ms | 1.473 | 0.672 | 接受 |
| v3：分块 → 整段 | 9.04 / 62.62 → 8.35 / 11.66 ms | 100.04 → 41.35 ms | 1.076 | 0.915 | 接受 |


## 相关阅读

完成 Chunked Prefill、分池与交接预算判断后，可以继续阅读异构 PD、真实服务调度和多实例验证。

- [DistServe 原论文：Disaggregating Prefill and Decoding for Goodput-optimized Large Language Model Serving](https://arxiv.org/abs/2401.09670)
- [vLLM 官方仓库](https://github.com/vllm-project/vllm)
- [SGLang PD Disaggregation 文档](https://docs.sglang.ai/advanced_features/pd_disaggregation.html)
- [39. Hetero PD and Serving Tiers | 异构 PD 与服务分层](./39_Hetero_PD_and_Serving_Tiers.ipynb)
- [70. Serving Scheduler Benchmark | 推理服务调度基准](./70_Serving_Scheduler_Benchmark.ipynb)